# C3 Spanish-Gap Wrap-Up — one notebook, run top to bottom

Closes out the last open item blocking IdiomBERT's Main-tier claim: the 10.6pp Hindi
Joint-F1 drop when Spanish enters the training combo (`en_hi_te` → `en_es_hi_te`,
System E joint mBERT).

**Two confounds were found in the original number, in order of how much they matter:**

1. **HP-tuning confound (bigger).** The fixed HPs (`lr=2e-5, cls_weight=0.3, span_weight=1.9`)
   were Optuna-swept on **English+Hindi+Telugu only** (`Sweep_Joint.py`, original
   `FIXED['langs']`). They were never re-tuned for the 4-language mix. `en_es_hi_te`'s
   drop may just mean its HPs are off-distribution for its own training data — not that
   Spanish is linguistically harmful to Hindi.
2. **Checkpoint-selection confound (smaller, orthogonal).** Best-epoch selection uses dev
   joint-F1 **pooled over all langs in the combo**. Once Spanish enters the pool, the
   selector's objective changes — it can trade Hindi performance for Spanish performance
   when picking a checkpoint, independent of any real cross-lingual interference.

**Run order — do not skip ahead:**
- Part 0: persistence gate (mandatory, always).
- Part 1: resweep HPs for `en_es_hi_te` on its own language mix.
- Part 2: retrain `en_es_hi_te` with its own tuned HPs, compare Hindi Joint F1 to the
  original 0.6357. **Read the verdict printed at the end of Part 2 before continuing** —
  it tells you whether Part 3 is even necessary.
- Part 3: checkpoint-selection confound test (pooled vs Hindi-only dev selection),
  only needed if Part 2's gap doesn't close.
- Part 4: status check for the other open GPU item (Sequential-BIO) — read-only, points
  to its own dedicated notebook if still pending.

Requires the patched `training/Train_Join.py`, `notebooks/colab/Language_Ablations/run_joint.py`,
and `notebooks/colab/Sweep_Joint.py` to be pushed to `main` before running Cell 1's `git pull`.

## Part 0 — Persistence gate (mandatory)

Two prior retrains on this repo were silently lost because output went to ephemeral
`/content` instead of Drive. The repo commits real `models/`+`results/` dirs, so a naive
symlink nests instead of replacing them. This cell rmtree's them first, then symlinks,
then hard-asserts the symlink resolved into Drive before anything else runs.

In [ ]:
import os, shutil, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')

# NOTE: the GitHub repo root IS the experiments dir — training/, notebooks/, models/
# sit directly at clone root. Do NOT append IdiomBERT/experiments.
REPO = '/content/Idiomator_Research'
if not os.path.isdir(f'{REPO}/.git'):
    r = subprocess.run(['git', 'clone', 'https://github.com/JustLetMeBeHello/Idiomator_Research.git', REPO],
                        capture_output=True, text=True)
    print(r.stdout, r.stderr)
    assert r.returncode == 0, f"git clone failed (exit {r.returncode}) — see stderr above, do not continue"
else:
    print(f"{REPO} already a git clone, reusing it")

os.chdir(REPO)
r = subprocess.run(['git', 'pull'], capture_output=True, text=True)
print(r.stdout, r.stderr)
assert r.returncode == 0, f"git pull failed (exit {r.returncode}) — see stderr above, do not continue"
print('cwd:', os.getcwd())

DRIVE = "/content/drive/MyDrive/Idiomator_Research"
for d in ["models", "results"]:
    os.makedirs(f"{DRIVE}/{d}", exist_ok=True)
    if os.path.islink(d):
        os.unlink(d)
    elif os.path.exists(d):
        shutil.rmtree(d)
    os.symlink(f"{DRIVE}/{d}", d)

for p in ("models", "results"):
    assert os.path.islink(p), f"{p} is a real dir, not a Drive symlink — writes will die on reset"
    assert "drive" in os.readlink(p).lower(), f"{p} symlink does not point at Drive: {os.readlink(p)}"
print("persist OK:", os.path.realpath("models"), os.path.realpath("results"))

# Confirm the patched files actually pulled — each file has its OWN marker
# (Sweep_Joint got --sweep_dir/--langs, not select_dev_lang).
patch_markers = {
    "training/Train_Join.py": "select_dev_lang",
    "notebooks/colab/Language_Ablations/run_joint.py": "select_dev_lang",
    "notebooks/colab/Sweep_Joint.py": "sweep_dir",
}
for f, marker in patch_markers.items():
    assert marker in open(f).read(), f"{f} missing marker '{marker}' — git pull did not update it"
print("patch check OK — all 3 files current")

In [ ]:
# Deps not preinstalled on Colab. optuna is needed for Part 1's HP sweep;
# transformers/torch/sklearn are already present in the Colab base image.
!pip install -q optuna

## Part 1 — Resweep HPs on `en_es_hi_te`'s own language mix

20-trial Optuna TPE sweep, same search space as the original, but `--langs English Spanish
Hindi Telugu` instead of the original English+Hindi+Telugu-only sweep. Writes to its own
`sweep_results_joint_sweep_en_es_hi_te/` — does not touch or require the original sweep's
(never-actually-run) output.

Cost: ~20 trials × a few epochs of 4-lang joint mBERT on T4 — budget a few hours.

In [ ]:
!python notebooks/colab/Sweep_Joint.py \
  --study_name joint_sweep_en_es_hi_te \
  --langs English Spanish Hindi Telugu \
  --n_trials 20

In [ ]:
import json
best_config = json.load(open('sweep_results_joint_sweep_en_es_hi_te/best_config.json'))
print(json.dumps(best_config, indent=2))

## Part 2 — Retrain `en_es_hi_te` with its own tuned HPs, compare to the old number

Writes to `joint_mbert_retuned/` (sibling dir) — the original `joint_mbert/` (old-HP,
Hindi joint F1 = 0.6357) is preserved for comparison, not overwritten.

In [ ]:
cfg = best_config['config']
OUT_DIR = 'models/en_es_hi_te/joint_mbert_retuned'

cmd = [
    sys.executable, '-u', 'training/Train_Join.py',
    '--output_dir',       OUT_DIR,
    '--data_dir',         'data/idioms_structured/Splits',
    '--langs',             'English', 'Spanish', 'Hindi', 'Telugu',
    '--seed',              '42',
    '--lr',                str(cfg['lr']),
    '--cls_loss_weight',   str(cfg['cls_loss_weight']),
    '--span_loss_weight',  str(cfg['span_loss_weight']),
    '--warmup_ratio',      str(cfg['warmup_ratio']),
    '--batch_size',        str(cfg['batch_size']),
    '--epochs',            str(cfg['epochs']),
    '--dropout',           str(cfg['dropout']),
]
subprocess.run(cmd, check=True)

In [ ]:
# Per-language Joint F1 from test_predictions.jsonl — same fields/method the canonical
# ablation_summary.csv is built from: joint_f1 = geomean(cls_macro_f1, mean(span_overlap_f1)),
# both computed per-language.
import json, math
from collections import defaultdict
from sklearn.metrics import f1_score

def per_lang_joint_f1(preds_path):
    rows = [json.loads(l) for l in open(preds_path)]
    by_lang = defaultdict(list)
    for r in rows:
        by_lang[r['language']].append(r)
    out = {}
    for lang, rs in by_lang.items():
        gold = [r['idiomaticity'] for r in rs]
        pred = [r['pred_idiomaticity'] for r in rs]
        cls_f1 = f1_score(gold, pred, average='macro')
        overlap = sum(r['span_overlap_f1'] for r in rs) / len(rs)
        joint = math.sqrt(cls_f1 * overlap) if cls_f1 > 0 and overlap > 0 else 0.0
        out[lang] = {'cls_macro_f1': cls_f1, 'span_overlap_f1': overlap, 'joint_f1': joint, 'n': len(rs)}
    return out

old = per_lang_joint_f1('models/en_es_hi_te/joint_mbert/test_predictions.jsonl')
new = per_lang_joint_f1('models/en_es_hi_te/joint_mbert_retuned/test_predictions.jsonl')
en_hi_te = per_lang_joint_f1('models/joint_mbert_en_hi_te/test_predictions.jsonl')

hi_old  = old['Hindi']['joint_f1']
hi_new  = new['Hindi']['joint_f1']
hi_ref  = en_hi_te['Hindi']['joint_f1']  # 0.7417 in the canonical CSV

print(f"en_hi_te      Hindi Joint F1 (no Spanish, reference) : {hi_ref:.4f}")
print(f"en_es_hi_te   Hindi Joint F1 (old default HPs)        : {hi_old:.4f}   drop = {hi_ref-hi_old:+.4f}")
print(f"en_es_hi_te   Hindi Joint F1 (retuned HPs)             : {hi_new:.4f}   drop = {hi_ref-hi_new:+.4f}")

residual = hi_ref - hi_new
closed_frac = 1 - (residual / (hi_ref - hi_old)) if (hi_ref - hi_old) != 0 else 0
print(f"\nGap closed by retuning: {closed_frac*100:.0f}%")
print()
if closed_frac > 0.6:
    print("VERDICT: mostly an HP-tuning artifact. Do NOT ship 'Spanish hurts Hindi' as a "
          "cross-lingual finding — reframe as an HP-sensitivity / methods note, or drop C3. "
          "Part 3 is optional (residual gap is small either way).")
elif closed_frac < 0.25:
    print("VERDICT: retuning barely moved it — HP-tuning was not the (main) cause. "
          "Proceed to Part 3 to test the checkpoint-selection confound before trusting "
          "the drop as real cross-lingual interference.")
else:
    print("VERDICT: partial effect — HP-tuning explains some but not all of the drop. "
          "Proceed to Part 3; use en_es_hi_te's RETUNED config (not the old defaults) "
          "as its arm's HPs so Part 3 isn't re-testing the same confound.")

## Part 3 — Checkpoint-selection confound test (only if Part 2 says so)

Each combo now trains with its **own** best HPs (en_hi_te's original defaults — the only
ones ever validated for it — vs en_es_hi_te's freshly-tuned config from Part 1). That's
correct practice, not a new confound.

What's still untested: pooled dev-selection changes objective composition when Spanish
enters the pool. `--select_dev_lang Hindi` removes that by picking the checkpoint on
Hindi's own dev score for both combos. Runs both arms × 3 seeds × 2 combos = 12 jobs.
Seed 42 pooled-arm results for `en_es_hi_te` already exist from Part 2 if closed_frac was
low — this reruns it anyway for a clean matched set; drop it manually if you want to save
one run.

In [ ]:
# HP sets — en_hi_te uses the script's own defaults (--lr etc. omitted = defaults),
# en_es_hi_te uses its Part-1 retuned config.
es_cfg = best_config['config']

def run_joint_job(combo, seed, select_dev_lang, use_retuned):
    cmd = [sys.executable, '-u', 'notebooks/colab/Language_Ablations/run_joint.py',
           '--only_combo', combo, '--seed', str(seed), '--force']
    if select_dev_lang:
        cmd += ['--select_dev_lang', select_dev_lang]
    if use_retuned:
        cmd += ['--lr', str(es_cfg['lr']),
                 '--cls_weight', str(es_cfg['cls_loss_weight']),
                 '--span_weight', str(es_cfg['span_loss_weight']),
                 '--epochs', str(es_cfg['epochs']),
                 '--batch', str(es_cfg['batch_size'])]
    print(f"\n### {combo} seed={seed} sel={select_dev_lang or 'pooled'} "
          f"hp={'retuned' if use_retuned else 'default'} ###", flush=True)
    subprocess.run(cmd, check=True)

SEEDS = [42, 123, 7]
for seed in SEEDS:
    for sel in [None, 'Hindi']:
        run_joint_job('en_hi_te',    seed, sel, use_retuned=False)
        run_joint_job('en_es_hi_te', seed, sel, use_retuned=True)

In [ ]:
# Read back all 12 runs' Hindi Joint F1, compare pooled vs Hindi-selected, mean±std across seeds.
import numpy as np

def joint_subdir(seed, sel):
    base = 'joint_mbert' if seed == 42 else f'joint_mbert_s{seed}'
    return base if not sel else f'{base}_dev{sel}'

rows = []
for combo in ['en_hi_te', 'en_es_hi_te']:
    for sel in [None, 'Hindi']:
        vals = []
        for seed in SEEDS:
            p = f"models/language_ablation_matrix/{combo}/{joint_subdir(seed, sel)}/test_predictions.jsonl"
            try:
                vals.append(per_lang_joint_f1(p)['Hindi']['joint_f1'])
            except FileNotFoundError:
                print(f"MISSING: {p}")
        if vals:
            rows.append((combo, sel or 'pooled', np.mean(vals), np.std(vals), vals))

print(f"{'combo':<14} {'selection':<10} {'mean':>8} {'std':>8}   seeds")
for combo, sel, mean, std, vals in rows:
    print(f"{combo:<14} {sel:<10} {mean:>8.4f} {std:>8.4f}   {[round(v,4) for v in vals]}")

print("\nRead this as: does the Hindi drop (en_hi_te vs en_es_hi_te) survive under")
print("Hindi-only selection at 3 seeds? If yes -> real interference, C3 stands, now")
print("confound-tested. If the gap shrinks toward the seed noise band under Hindi-only")
print("selection -> the residual drop was a pooled-selection artifact, not interference.")

## Part 4 — Sequential-BIO status check (the other open GPU item)

Read-only. Does not train anything. Per the 2026-06-30 handoff, this experiment
(completes the 2×2 matrix: Single-pass/Sequential × QA/BIO) was built and pushed but the
Colab run itself was never confirmed to have happened. This cell just checks whether it's
already done.

In [ ]:
import os
p = 'models/sequential_bio/aggregate_results.json'
if os.path.exists(p):
    print(f"FOUND — {p} exists, already done. Register its numbers in key_numbers.md if not done yet.")
    print(open(p).read()[:1000])
else:
    print(f"NOT FOUND — {p} does not exist yet.")
    print("Run notebooks/colab/Sequential_BIO_Training.ipynb (Cells 1-5) separately — ")
    print("not duplicated in this notebook since it's an independent experiment (not part ")
    print("of the C3 Spanish-gap thread this notebook exists to close out).")

## Not covered here

- **Telugu IAA** — human annotation work (recruiting a second independent native Telugu
  annotator), not a GPU/notebook task. Separate track, gates MultiIdiom's tier.
- **`en_hi_te`'s own HPs were also never confirmed via a saved sweep output** (no
  `best_config.json` for the original En+Hi+Te sweep exists anywhere in the repo — the
  current defaults may just be hand-set, not Optuna's actual winner). Part 3 above treats
  them as "the only ones ever validated for it" out of necessity, not because they're
  confirmed optimal. If you want full symmetry, resweep `en_hi_te` too
  (`--langs English Hindi Telugu --study_name joint_sweep_en_hi_te`) before trusting a
  precise magnitude on the residual gap — flagged, not resolved, here.